# Movie Rental Shop Data Analysis

## About the Data
This data comes from a DVD and Movie Rental shop (like Blockbuster). It tracks the daily operations of the stored including what movies exist, which customers rented them, and the payments made.

Here is a quick overview of what each file contains:
- **actor.csv**: Information about movie actors
- **address.csv**: Street addresses for customers and stores
- **category.csv**: The genres of the movies (e.g., Action, Comedy)
- **city.csv / country.csv**: Geographic data mapping addresses to cities and countries
- **customer.csv**: Details of the people who rent the movies
- **film.csv**: Master list of all movies available for rent
- **film_actor.csv / film_category.csv**: Linking tables to connect films with actors and genres
- **inventory.csv**: The physical copies of the films kept in the stores
- **language.csv**: The spoken language of the movies
- **payment.csv**: The financial transactions made by customers for rentals
- **rental.csv**: The operational events of when a DVD was checked out and returned
- **staff.csv**: Information about the employees working at the stores
- **store.csv**: Details identifying each individual store location



## 1. Import All CSV Files
First, we will use pandas to load all of our CSV files into data variables. We use simple names so it is very easy to read. Note that we do not print any extra text here, keeping the notebook clean.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Loading all tables directly
actor = pd.read_csv("actor.csv")
address = pd.read_csv("address.csv")
category = pd.read_csv("category.csv")
city = pd.read_csv("city.csv")
country = pd.read_csv("country.csv")
customer = pd.read_csv("customer.csv")
film = pd.read_csv("film.csv")
film_actor = pd.read_csv("film_actor.csv")
film_category = pd.read_csv("film_category.csv")
inventory = pd.read_csv("inventory.csv")
language = pd.read_csv("language.csv")
payment = pd.read_csv("payment.csv")
rental = pd.read_csv("rental.csv")
staff = pd.read_csv("staff.csv")
store = pd.read_csv("store.csv")


## 2. Basic Inspection
Let's see how many rows are in our core tables to understand the size of our store's activity.
Afterward, we will look at the first few rows of the main financial table (`payment`) and the operations table (`rental`).


In [ ]:
# Let's verify the row counts of our most important tables
print(f"Total Customers: {customer.shape[0]}")
print(f"Total Movies: {film.shape[0]}")
print(f"Total Rental Events: {rental.shape[0]}")
print(f"Total Payments Made: {payment.shape[0]}")


In [ ]:
# Look at the payment data
payment.head()


In [ ]:
# Look at the rental data
rental.head()


### Checking for Missing Values
Let's check if there are empty rows in the payment table. Missing data can throw off our revenue calculations!


In [ ]:
# This sums up the empty values in each column
payment.isnull().sum()


## 3. Data Cleaning
We need to fix our date columns. Dates are currently stored as plain text. By converting them into real time-based data structures (datetime), we gain the ability to group our analysis by days, months, and years.

We will also drop a database metadata column named `last_update` because it provides no value to our business analysis.


In [ ]:
# Convert text dates to real datetime logic
payment['payment_date'] = pd.to_datetime(payment['payment_date'])
rental['rental_date'] = pd.to_datetime(rental['rental_date'])
rental['return_date'] = pd.to_datetime(rental['return_date'])

# Drop useless columns from payment table to clean it up
payment = payment.drop(columns=['last_update'], errors='ignore')

# Show that the conversion worked
print("Data Types in Payment Table:")
print(payment.dtypes)


## 4. Feature Engineering
Feature Engineering means creating new, more useful data columns from the ones we already have. 

Here we will create:
1. **The month the payment happened in**: Useful for tracing revenue over time.
2. **The number of days the movie was rented for**: Useful to find out how long people keep movies.


In [ ]:
# Extract the period/month from the actual payment date
payment['payment_month'] = payment['payment_date'].dt.to_period('M')

# Calculate how many days a movie was rented
rental['rental_duration_days'] = (rental['return_date'] - rental['rental_date']).dt.days

# Show the new columns
print("New columns created successfully!")


## 5. Merging Tables (Creating our Master Table)
In relational databases, information is broken up into small connecting pieces. To figure out who bought what film, and what category that film belonged to, we have to stitch it all together.

We will create a massive `master_table` step-by-step.

### Merge Step 1: Link Payments to Rentals
We will take the `payment` table and merge the `rental` table into it.
- **Key used**: `rental_id`
- **Why**: To connect the actual money spent to the specific operational checkout event.
- **New info**: Adds `rental_date`, `return_date`, and `inventory_id` to the payment details.


In [ ]:
# Merge 1
master_table = pd.merge(payment, rental, on='rental_id', how='left')
master_table.head(3)


### Merge Step 2: Link Rentals to Physical Inventory
Now we link the updated master table to the `inventory` table.
- **Key used**: `inventory_id`
- **Why**: To connect the operational checkout event to the physical DVD in the store.
- **New info**: Adds `film_id`, which is a stepping stone to find the movie title.


In [ ]:
# Merge 2
master_table = pd.merge(master_table, inventory, on='inventory_id', how='left')


### Merge Step 3: Link Inventory to Film Details
Now we bring in the `film` table.
- **Key used**: `film_id`
- **Why**: Since we know WHICH physical DVD it was, we can finally grab the movie's title and description.
- **New info**: Adds `title`, `rental_rate`, `rating`, etc.


In [ ]:
# Merge 3
master_table = pd.merge(master_table, film[['film_id', 'title']], on='film_id', how='left')


### Merge Step 4: Link Payment to the Customer
Let's add the customer's name so we know precisely who is renting the films!
- **Key used**: `customer_id` (specifically, we use `customer_id_x` because earlier merges created duplicates. We'll simply use the base customer_id columns.)
Wait, both payment and rental tables have customer_id! Let's clean that up by merging cleanly with the original customer_id from the payment.


In [ ]:
# Merge 4
master_table = pd.merge(master_table, customer[['customer_id', 'first_name', 'last_name']], on='customer_id', how='left')

# Let's create a full name column for convenience
master_table['customer_name'] = master_table['first_name'] + " " + master_table['last_name']


### Merge Step 5 & 6: Link to the Movie Category
Finally, we want to know what genre the movie was. This takes two hops. Film links to `film_category`, and `film_category` links to `category`.
- **Key used**: `film_id` and then `category_id`
- **Why**: To group our movies by Action, Sci-Fi, Drama, etc.
- **New info**: Adds the category `name`.


In [ ]:
# Merge 5 & 6
master_table = pd.merge(master_table, film_category, on='film_id', how='left')
master_table = pd.merge(master_table, category[['category_id', 'name']], on='category_id', how='left')

# Rename the category name column so it makes more sense
master_table = master_table.rename(columns={'name': 'category_name'})

print(f"Master table has {master_table.shape[0]} rows and {master_table.shape[1]} columns!")


## 6. Key Performance Indicators (KPIs)
Now that we have all the data in one place, let's calculate the top-level metrics for the store.


In [ ]:
# Total Revenue
total_revenue = master_table['amount'].sum()

# Total count of successful payments/rentals
total_rentals = master_table['payment_id'].count()

# Average amount spent per transaction
average_payment = master_table['amount'].mean()

print(f"Top Level KPIs")
print(f"----------------------")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Rentals Processed: {total_rentals:,}")
print(f"Average Revenue per Rental: ${average_payment:,.2f}")


### Top Performing Entities
Let's find out which movie made the most money!


In [ ]:
# Grouping by the movie title and summing the money spent
top_movies = master_table.groupby('title')['amount'].sum().sort_values(ascending=False).head(5)

print("\nTop 5 Highest Earning Movies:")
print(top_movies)


## 7. Charts and Visualizations
Visualizing numbers helps people understand them much quicker. We will use `matplotlib` and `seaborn`.


### Category Revenue Comparison
Let's see which movie genres bring in the most money.


In [ ]:
plt.figure(figsize=(10, 5))
revenue_by_cat = master_table.groupby('category_name')['amount'].sum().sort_values(ascending=False)

# Make a barplot
sns.barplot(x=revenue_by_cat.values, y=revenue_by_cat.index, palette='viridis', hue=revenue_by_cat.index, legend=False)
plt.title('Total Revenue by Movie Category')
plt.xlabel('Total Revenue ($)')
plt.ylabel('Category')
plt.show()


**Insights:**
1. Action & Sports bring in the absolute most revenue for the store.
2. Less mainstream categories like Music and Thriller fall lower on the list.


### Monthly Revenue Trend
Let's check if the business is growing or shrinking month over month.


In [ ]:
plt.figure(figsize=(8, 4))
# Convert our 'period' month to string so the plot likes it
monthly_rev = master_table.groupby(master_table['payment_month'].astype(str))['amount'].sum()

plt.plot(monthly_rev.index, monthly_rev.values, marker='o', color='green', linewidth=2)
plt.title('Monthly Store Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


**Insights:**
1. A clear visual trend line proves the trajectory of the shop.
2. Depending on the peaks, we could match these trends to high season holidays or new blockbusters.


## 8. Final Summary
**What this data represents**: This is a robust slice of information reflecting the daily operation, financial success, and inventory tracking of a physical movie rental chain.

**Our most important result**: By linking six disparate tables together, we successfully reconstructed full narrative journeys tracing a dollar spent directly to a customer's name and the specific genre of movie they were holding. 

**Key Business Insights**: 
1. The company should prioritize stocking extra copies of "Sports" and "Sci-Fi" movies as they drive extreme revenue. 
2. The business averages about $4.20 per checkout, which suggests customers typically rent single items for short durations rather than bulk renting.
